In [0]:
jan2016_1_df = spark.read.format("csv").option("header", True).load("/Volumes/workspace/tfl/raw/01aJourneyDataExtract10Jan16-23Jan16.csv")

In [0]:
jan2016_1_df.printSchema()

In [0]:
rename_dict = {
    'Rental Id': 'rental_id',
    'Duration': 'duration',
    'Bike Id': 'bike_id',
    'End Date': 'end_date',
    'EndStation Id': 'end_station_id',
    'EndStation Name': 'end_station_name',
    'Start Date': 'start_date',
    'StartStation Id': 'start_station_id',
    'StartStation Name': 'start_station_name'
}

def rename_columns(df, rename_dict):
    for old_name, new_name in rename_dict.items():
        df = df.withColumnRenamed(old_name, new_name)
    return df

In [0]:
from pyspark.sql.functions import lit

files = ["01aJourneyDataExtract10Jan16-23Jan16.csv", 
         "01bJourneyDataExtract24Jan16-06Feb16.csv", 
         "02aJourneyDataExtract07Fe16-20Feb2016.csv", 
         "02bJourneyDataExtract21Feb16-05Mar2016.csv", 
         "03JourneyDataExtract06Mar2016-31Mar2016.csv",
         "04JourneyDataExtract01Apr2016-30Apr2016.csv",	
         "05JourneyDataExtract01May2016-17May2016.csv",	
         "06JourneyDataExtract18May2016-24May2016.csv",
         "07JourneyDataExtract25May2016-31May2016.csv"]

dfs = []

for file in files:
    df = rename_columns(spark.read.format("csv").option("header", True).load("/Volumes/workspace/tfl/raw/" + file), rename_dict)
    df = df.withColumn("source_file", lit("/Volumes/workspace/tfl/raw/" + file))
    dfs.append(df)

# join all data into one df
named_df = dfs[0]
for i in range(1, len(dfs)):
    named_df = named_df.unionByName(dfs[i], allowMissingColumns=True)
named_df.printSchema()

In [0]:
# get rows with values in _c9, _c10 or _c11 columns
named_df.filter((named_df["_c9"].isNotNull()) | (named_df["_c10"].isNotNull()) | (named_df["_c11"].isNotNull())).show()


In [0]:
from pyspark.sql import functions as F

# add column with ingested date
bronze_df = named_df.withColumn(
    "ingested_at", F.current_timestamp()
)

bronze_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.tfl.bronze_journeys"
)

In [0]:
bronze = spark.read.table("workspace.tfl.bronze_journeys")
bronze.count()

In [0]:
bronze.printSchema()

In [0]:
bronze.show(15)